### Basic Setups

In [7]:
import os
import cv2
import matplotlib.pyplot as plt
import random
from ultralytics import YOLO
from pathlib import Path
import shutil

### Data Preprocessing

In [8]:
# --- CONFIGURATION ---
DATASET_DIR = Path("Body Focused Dataset")  # Path to your main dataset folder
IMAGES_SUBDIR = "train/images"       # Name of the images folder
LABELS_SUBDIR = "train/labels"       # Name of the labels folder

# Target ratios (Must add up to 1.0)
TRAIN_RATIO = 0.75
VAL_RATIO = 0.20
TEST_RATIO = 0.05  # Adjusted from 0.05 to ensure 100% collection

def split_yolo_dataset():
    img_dir = DATASET_DIR / IMAGES_SUBDIR
    lbl_dir = DATASET_DIR / LABELS_SUBDIR

    if not img_dir.exists() or not lbl_dir.exists():
        raise FileNotFoundError(
            f"Could not find '{IMAGES_SUBDIR}' or '{LABELS_SUBDIR}' inside {DATASET_DIR}. "
            "Please check your directory names."
        )

    # Gather all unique file stems based on images (handles .jpg, .png, etc.)
    image_extensions = {".jpg", ".jpeg", ".png", ".bmp"}
    all_stems = [f.stem for f in img_dir.iterdir() if f.suffix.lower() in image_extensions]
    
    # Shuffle randomly to mix elderly and non-elderly classes evenly
    random.seed(42)
    random.shuffle(all_stems)

    total_files = len(all_stems)
    if total_files == 0:
        print("No images found to split!")
        return

    # Calculate split indices
    train_end = int(total_files * TRAIN_RATIO)
    val_end = train_end + int(total_files * VAL_RATIO)

    splits = {
        "train": all_stems[:train_end],
        "val": all_stems[train_end:val_end],
        "test": all_stems[val_end:]
    }

    print(f"Total files found: {total_files}")
    print(f"Splitting into -> Train: {len(splits['train'])}, Val: {len(splits['val'])}, Test: {len(splits['test'])}")

    # Create target directories
    for split in ["train", "val", "test"]:
        (DATASET_DIR / split / IMAGES_SUBDIR).mkdir(parents=True, exist_ok=True)
        (DATASET_DIR / split / LABELS_SUBDIR).mkdir(parents=True, exist_ok=True)

    # Move files to their respective split directories
    for split, stems in splits.items():
        for stem in stems:
            # Find the actual image file with its extension
            img_file = next(img_dir.glob(f"{stem}.*"), None)
            lbl_file = lbl_dir / f"{stem}.txt"

            if img_file and lbl_file.exists():
                # Destination paths
                dest_img = DATASET_DIR / split / IMAGES_SUBDIR / img_file.name
                dest_lbl = DATASET_DIR / split / LABELS_SUBDIR / lbl_file.name

                # Move files
                shutil.move(str(img_file), str(dest_img))
                shutil.move(str(lbl_file), str(dest_lbl))
            else:
                print(f"Warning: Missing pair for file stem: {stem}")

    # Clean up original top-level images and labels folders if empty
    try:
        img_dir.rmdir()
        lbl_dir.rmdir()
        print("Successfully reorganized dataset folders!")
    except OSError:
        print("Reorganization complete, but original folders were not entirely empty.")

if __name__ == "__main__":
    split_yolo_dataset()

Total files found: 848
Splitting into -> Train: 636, Val: 169, Test: 43
Successfully reorganized dataset folders!


### Training Process

In [10]:
model = YOLO('yolov8n.pt')

model.info(detailed=True)

WARNING Download failure, retrying 1/3 https://github.com/ultralytics/assets/releases/download/v8.3.0/yolov8n.pt... HTTP Error 502: Bad Gateway or Proxy Error


FileNotFoundError: [Errno 2] No such file or directory: 'yolov8n.pt'

In [14]:
from ultralytics import YOLO

DataPath = 'Body Focused Dataset/data.yaml'

# ==========================================
# STAGE 1 : WARMUP 
# ==========================================
model = YOLO('yolov8n.pt') 

model.train(
    data         =    DataPath, 
    epochs       =    15,          
    freeze       =    22, 
    lr0          =    0.01,
    project      =    'Detector Result',
    name         =    'Warmup',
    device       =    0,
    imgsz        =    640,         
    amp          =    True,
    batch        =    8,  
    workers      =    2,
    cache        =    False,      
    patience     =    10,       
    exist_ok     =    True,
    
    # --- AUGMENTATION ---
    cls          =     4.0,           
    box          =     7.5, 
    hsv_v        =     0.4,
    scale        =     0.3,
    fliplr       =     0.5,
    mosaic       =     1.0,
    overlap_mask =     True  
)

New https://pypi.org/project/ultralytics/8.4.57 available  Update with 'pip install -U ultralytics'
Ultralytics 8.3.222  Python-3.10.18 torch-2.5.1 CUDA:0 (NVIDIA GeForce RTX 4060 Laptop GPU, 8188MiB)
engine\trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=4.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=Body Focused Dataset/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=15, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=22, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=Warmup, nbs=64, nms=False

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x000002931F104940>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.0

In [15]:
# ==========================================
# STAGE 2 : FINE-TUNE
# ==========================================
model = YOLO('Detector Result/Warmup/weights/best.pt') 

model.train(
    data         =    DataPath, 
    epochs       =    50,          
    freeze       =    0,             
    lr0          =    0.001,
    cos_lr       =    True,      
    project      =    'Detector Result',
    name         =    'Fine Tune',
    device       =    0,
    imgsz        =    640,        
    amp          =    True,
    batch        =    8,
    workers      =    2,
    cache        =    False,      
    patience     =    15,
    exist_ok     =    True,
    
    # --- AUGMENTATION ---
    cls          =    5.0,           
    box          =    10.0,          
    hsv_v        =    0.4,
    scale        =    0.4,  
    mosaic       =    1.0,
    close_mosaic =    20,                
    fliplr       =    0.5
)

New https://pypi.org/project/ultralytics/8.4.57 available  Update with 'pip install -U ultralytics'
Ultralytics 8.3.222  Python-3.10.18 torch-2.5.1 CUDA:0 (NVIDIA GeForce RTX 4060 Laptop GPU, 8188MiB)
engine\trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=10.0, cache=False, cfg=None, classes=None, close_mosaic=20, cls=5.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=Body Focused Dataset/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=0, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=Detector Result/Warmup/weights/best.pt, momentum=0.937, mosaic=1.0, multi_scale=False, na

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x000002932AEF69B0>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.0

### Testing I

In [ ]:
ModelTest = YOLO('Detector Result/Fine Tune/weights/best.pt')

TestImageFolder = 'Body Focused Dataset/test/images'
TestImages = [os.path.join(TestImageFolder, f) for f in os.listdir(TestImageFolder) if f.endswith(('.jpg', '.png', '.jpeg'))]

SampleImages = random.sample(TestImages, 6)

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for i, ImagePath in enumerate(SampleImages):
    results = ModelTest.predict(source=ImagePath, conf=0.2, save=False)[0]
    
    res_plotted = results.plot()
    
    res_rgb = cv2.cvtColor(res_plotted, cv2.COLOR_BGR2RGB)
    
    axes[i].imshow(res_rgb)
    axes[i].set_title(f"Test Image {i+1}")
    axes[i].axis('off')

plt.tight_layout()
plt.show()

ModelTest.predict(source=TestImageFolder, conf=0.2, save=True, project='Detector Result', name='Testing I Visuals')
print(f"All annotated images saved to: Detector Result/Testing I Visuals")


image 1/1 c:\Users\justi\Documents\Project\SoCS\Python\AI Solutions\Body Focused Dataset\test\images\Thor284_jpg.rf.2b75e2bb3bfc6cbd50df2af4ba774161.jpg: 640x384 1 muda, 35.3ms
Speed: 2.2ms preprocess, 35.3ms inference, 7.7ms postprocess per image at shape (1, 3, 640, 384)

image 1/1 c:\Users\justi\Documents\Project\SoCS\Python\AI Solutions\Body Focused Dataset\test\images\BF85-WF3_png.rf.35b40ccc41278126d5bf0c81bb55d35d.jpg: 448x640 1 muda, 33.4ms
Speed: 3.9ms preprocess, 33.4ms inference, 1.3ms postprocess per image at shape (1, 3, 448, 640)

image 1/1 c:\Users\justi\Documents\Project\SoCS\Python\AI Solutions\Body Focused Dataset\test\images\Mora596_jpg.rf.f700c67385d9fd14ba045a72e4f95316.jpg: 640x352 1 muda, 32.6ms
Speed: 1.9ms preprocess, 32.6ms inference, 1.2ms postprocess per image at shape (1, 3, 640, 352)

image 1/1 c:\Users\justi\Documents\Project\SoCS\Python\AI Solutions\Body Focused Dataset\test\images\9cf5f651bdafc67be0391cd4a8e511b24d714377_jpg.rf.4bc1274fb81a335f1e2d4f27

<Figure size 1800x1000 with 6 Axes>


image 1/43 c:\Users\justi\Documents\Project\SoCS\Python\AI Solutions\Body Focused Dataset\test\images\0d29264ace6fc7eac16586c063d22406ab3c4d97_jpg.rf.f45c45a75b33238a5975a3c71c1cd08d.jpg: 640x480 1 lansia, 72.8ms
image 2/43 c:\Users\justi\Documents\Project\SoCS\Python\AI Solutions\Body Focused Dataset\test\images\1d8fd0f1de12f2d881fb9e04ea2188a9cc92d22e_jpg.rf.ef953a72516020d8cb8ad4f0a225fd9d.jpg: 640x448 1 lansia, 33.1ms
image 3/43 c:\Users\justi\Documents\Project\SoCS\Python\AI Solutions\Body Focused Dataset\test\images\1f66eb01015318e005850c19a6b97f740c0d8fd1_jpg.rf.39acde1c16cabee0e25baa5ac9e4a648.jpg: 640x448 1 lansia, 15.7ms
image 4/43 c:\Users\justi\Documents\Project\SoCS\Python\AI Solutions\Body Focused Dataset\test\images\24f9f384a1c16c415a4e5b9ed88b047e80d20758_jpg.rf.9a7682874b11f5512a7f4f3129fb838b.jpg: 640x480 1 lansia, 19.5ms
image 5/43 c:\Users\justi\Documents\Project\SoCS\Python\AI Solutions\Body Focused Dataset\test\images\26dcda61312fee85da776b7537b9c5fb82ef02f9_jpg.

### Testing II

In [20]:
MODEL_PATH = 'Detector Result/Fine Tune/weights/best.pt'
VIDEO_PATH = 'Test II Data.mp4'

def process_video():
    print("[INFO] Loading model...")
    model = YOLO(MODEL_PATH)

    print("[INFO] Processing portrait video...")
    # YOLO automatically reads phone orientation metadata, keeps aspect ratio, 
    # and outputs a fully annotated video file.
    results = model.predict(
        source=VIDEO_PATH,
        imgsz=640,
        conf=0.4,           
        save=True,         
        project='Detector Result',
        name='Testing II Visuals',
        exist_ok=True,
        device=0            # Use your GPU
    )
    
    print("[INFO] Done! Check the 'Detector Result/Testing II Visuals/Test Result' folder for your output.")

if __name__ == "__main__":
    process_video()

[INFO] Loading model...
[INFO] Processing portrait video...

WARNING 
inference results will accumulate in RAM unless `stream=True` is passed, causing potential out-of-memory
errors for large sources or long-running streams and videos. See https://docs.ultralytics.com/modes/predict/ for help.

Example:
    results = model(source=..., stream=True)  # generator of Results objects
    for r in results:
        boxes = r.boxes  # Boxes object for bbox outputs
        masks = r.masks  # Masks object for segment masks outputs
        probs = r.probs  # Class probabilities for classification outputs

video 1/1 (frame 1/217) c:\Users\justi\Documents\Project\SoCS\Python\AI Solutions\Test II Data.mp4: 640x352 (no detections), 17.9ms
video 1/1 (frame 2/217) c:\Users\justi\Documents\Project\SoCS\Python\AI Solutions\Test II Data.mp4: 640x352 (no detections), 16.3ms
video 1/1 (frame 3/217) c:\Users\justi\Documents\Project\SoCS\Python\AI Solutions\Test II Data.mp4: 640x352 (no detections), 16.0ms
vid